# P05 — Analyse et prévision de la régularité TGV (AQST)

**Contexte métier.** SNCF Voyageurs et AQST (autorité de qualité de service) : suivre la régularité par liaison et par mois, anticiper les mois à risque, hiérarchiser les causes de retard. Enjeu contractuel : 2026 se conclut autour de 84 % de trains à l'heure.

**Objectif de l'EDA.** Construire la cible (taux de régularité mensuel par liaison) et identifier les régresseurs (mois, liaison, causes de retard).

**Données.** Régularité mensuelle TGV toutes liaisons — SNCF via data.sncf.com. Licence ouverte Etalab 2.0. 12 834 observations, 130 liaisons, 2018-2026.

## 1. Téléchargement

Export CSV de l'open data SNCF (portail Opendatasoft) :

In [ ]:
import urllib.request, os
os.makedirs("data", exist_ok=True)

URL = "https://data.sncf.com/api/explore/v2.1/catalog/datasets/regularite-mensuelle-tgv-aqst/exports/csv?use_labels=true"
if not os.path.exists("data/regularite-mensuelle-tgv-aqst.csv"):
    urllib.request.urlretrieve(URL, "data/regularite-mensuelle-tgv-aqst.csv")
print(os.path.getsize("data/regularite-mensuelle-tgv-aqst.csv") / 1e6, "Mo")

## 2. Chargement et construction de la cible

Séparateur `;`, décimales à virgule. Le taux de régularité n'existe pas directement : il se calcule comme (prévues − annulées − retards) / prévues.

In [ ]:
import pandas as pd, numpy as np

df = pd.read_csv("data/regularite-mensuelle-tgv-aqst.csv", sep=";", low_memory=False)
df.columns = [c.strip() for c in df.columns]
df = df.dropna(subset=["Nombre de circulations prévues"])
df = df[df["Nombre de circulations prévues"] > 0]
ann = pd.to_numeric(df["Nombre de trains annulés"], errors="coerce").fillna(0)
ret = pd.to_numeric(df["Nombre de trains en retard à l'arrivée"], errors="coerce").fillna(0)
df["taux_regularite"] = 100 * (df["Nombre de circulations prévues"] - ann - ret) / df["Nombre de circulations prévues"]
df["annee"] = df["Date"].str[:4]
df["mois"] = df["Date"].str[5:7]
df["liaison"] = df["Gare de départ"] + " -> " + df["Gare d'arrivée"]
print(len(df), "obs,", df["liaison"].nunique(), "liaisons")

## 3. Analyse

Distribution du taux, trajectoire annuelle, saisonnalité mensuelle, liaisons extrêmes et part des causes de retard.

In [ ]:
print(df["taux_regularite"].describe(percentiles=[.05, .5, .95]).round(1).to_dict())
print()
print("Taux moyen par année :", df.groupby("annee")["taux_regularite"].mean().round(1).to_dict())
print("Taux moyen par mois :", df.groupby("mois")["taux_regularite"].mean().round(1).to_dict())
print()
g = df.groupby("liaison")["taux_regularite"].agg(["mean", "count"])
g = g[g["count"] >= 12]
print("Pires liaisons :", g.sort_values("mean").head(5).round(1).to_dict("index"))
print("Causes de retard (moyenne des pourcentages) :")
for c in [c for c in df.columns if c.startswith("Prct")]:
    print(" ", c, round(pd.to_numeric(df[c], errors="coerce").mean(), 1))

## 4. Résultats observés (exécution du 16/10/2026)

- n lignes : 12834
- periode : 2018-01 → 2026-09
- n liaisons : 130
- taux regularite stats : count : 12 834.00, mean : 82.00, std : 10.80, min : 0.00, 5% : 61.10, 25% : 77.90, 50% : 84.50, 75% : 89.00, 95% : 93…
- taux moyen par annee : 2018 : 76.30, 2019 : 82.80, 2020 : 79.50, 2021 : 84.90, 2022 : 84.00, 2023 : 82.00, 2024 : 84.90, 2025 : 83.60, 2026 …
- taux moyen par mois : 01 : 83.30, 02 : 85.00, 03 : 83.50, 04 : 75.90, 05 : 81.70, 06 : 79.80, 07 : 79.40, 08 : 84.50, 09 : 85.00, 10 : 83.60
- pire liaisons taux moyen : ITALIE -> PARIS LYON : 65.10, CHAMBERY CHALLES LES EAUX -> PARIS LYON : 70.10, LYON PART DIEU -> LILLE : 70.50, MACON…
- meilleures liaisons taux moyen : RENNES -> PARIS VAUGIRARD : 91.40, PARIS LYON -> LYON PART DIEU : 89.20, BESANCON FRANCHE COMTE TGV -> PARIS LYON : 8…
- pct retards par cause : Prct retard pour causes externes : 22.30, Prct retard pour cause infrastructure : 21.20, Prct retard pour cause gesti…
- corr causes regularite : Prct retard pour causes externes : -0.05, Prct retard pour cause infrastructure : 0.01, Prct retard pour cause gestio…

## 5. Variable cible et régresseurs

**Cible (régression) : `taux_regularite` mensuel par liaison** (ou classification "mois à risque" < 80 % si l'on préfère un problème binaire).

**Régresseurs retenus :**
- `liaison` (gare de départ × gare d'arrivée) : la variabilité dominante (65,1 % pour l'Italie→Paris contre 91,4 % pour Rennes→Paris)
- `mois` : avril et juillet sont les pires mois (75,9 % et 79,4 %), septembre-octobre les meilleurs (85 %)
- `annee` (tendance, effet COVID 2020 : 79,5 %)
- `Durée moyenne du trajet` et volumes (`Nombre de circulations prévues`)

**Pièges :**
- Le taux de régularité n'est pas fourni : il se calcule — attention aux divisions par zéro et aux lignes de très faible volume (une annulation = −100 %)
- Les colonnes `Prct retard pour causes…` décrivent les causes constatées : corrélées à la cible mais disponibles après coup ; corr. faibles (−0,05 à +0,05) avec le taux — à utiliser en analyse, pas en prévision
- Le COVID (2020-2021) structure le dataset : le garder ou l'exclure doit être un choix explicite
- Ne pas laisser les causes de retard fuiter dans un modèle prédictif